# Preprocessing Teks Ulasan Produk E-Commerce (Bahasa Indonesia)

**Nama:** Muhammad Hafidz  
**NIM:** 24146028  
**Mata Kuliah:** Pengolahan Bahasa Alami (NLP)  
**Dataset:** [dipawidia/ecommerce-product-reviews-sentiment](https://huggingface.co/datasets/dipawidia/ecommerce-product-reviews-sentiment)  
**Repo GitHub:** https://github.com/MuhammadHafidz212-cell/nlp

Notebook ini membangun pipeline *preprocessing* teks ulasan e-commerce berbahasa Indonesia
secara bertahap: **case folding -> cleaning -> tokenisasi -> stopword removal -> stemming**.
Setiap tahap disimpan pada kolomnya sendiri sehingga prosesnya transparan dan dapat diaudit.
Di akhir, notebook menyimpan `data/hasil_preprocessing.csv` dan `data/statistik.json`
yang menjadi sumber angka tunggal untuk laporan PDF.

In [1]:
import os, re, json, random, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
FIG_DIR = BASE_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

print("Working dir :", BASE_DIR)
print("Data dir    :", DATA_DIR)
print("Figure dir  :", FIG_DIR)

Working dir : D:\NLP\preprocessing-ulasan-ecommerce-id
Data dir    : D:\NLP\preprocessing-ulasan-ecommerce-id\data
Figure dir  : D:\NLP\preprocessing-ulasan-ecommerce-id\figures


## 1. Load Dataset

Dataset dimuat dari Hugging Face menggunakan `datasets.load_dataset`. Apabila cara tersebut
gagal (misalnya karena masalah jaringan atau versi pustaka), notebook otomatis memakai
alternatif `huggingface_hub` untuk mengunduh berkas Parquet/CSV lalu membacanya dengan pandas.
Nama kolom **tidak ditebak**: kolom aktual ditampilkan dan diperiksa sebelum diproses.

In [2]:
DATASET_NAME = "dipawidia/ecommerce-product-reviews-sentiment"
df = None
load_method = None

try:
    from datasets import load_dataset
    ds = load_dataset(DATASET_NAME)
    split = "train" if "train" in ds else list(ds.keys())[0]
    df = ds[split].to_pandas()
    load_method = f"datasets.load_dataset (split='{split}')"
    print("Berhasil memuat via datasets:", load_method)
except Exception as e:
    print("load_dataset gagal:", repr(e))
    print("Mencoba fallback huggingface_hub + pandas ...")
    from huggingface_hub import list_repo_files, hf_hub_download
    files = list_repo_files(DATASET_NAME, repo_type="dataset")
    candidate = [f for f in files if f.endswith(".parquet")] or [f for f in files if f.endswith(".csv")]
    if not candidate:
        raise RuntimeError("Tidak ditemukan berkas parquet/csv pada repo dataset: " + str(files))
    frames = []
    for f in candidate:
        p = hf_hub_download(DATASET_NAME, f, repo_type="dataset")
        frames.append(pd.read_parquet(p) if p.endswith(".parquet") else pd.read_csv(p))
    df = pd.concat(frames, ignore_index=True)
    load_method = "huggingface_hub + pandas (" + ", ".join(candidate) + ")"
    print("Berhasil memuat via fallback:", load_method)

df = df.reset_index(drop=True)
print("Jumlah baris :", len(df))
print("Kolom        :", list(df.columns))
display(df.head())

Berhasil memuat via datasets: datasets.load_dataset (split='train')
Jumlah baris : 11606
Kolom        : ['review', 'translate', 'sentimen']


,review,translate,sentimen
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,"sound and mic satisfied. I only ordered the one that was on and I made sure that it clicked again, but the one that ...",0
1,Good. Mudah-mudahan awet. Makasih,good. hopefully durable. Thanks,1
2,"kondisi dus robek2, tolong di perhatikan lagi","the condition of the box is torn, please pay attention again",0
3,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo ...,"product quality is good. affordable prices and very fast delivery. I've already ordered here, it's okay, but unfortu...",0
4,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit","overall is good, the buttons all work, but the packaging is lacking because it's dented and torn a little",0


## 2. Informasi Dataset

Pada bagian ini ditampilkan jumlah data, kolom beserta tipe datanya, jumlah *missing value*,
jumlah baris duplikat, serta distribusi label sentimen dalam jumlah dan persentase.
Label mengikuti dokumentasi dataset: **0 = negatif**, **1 = positif**.
Grafik distribusi disimpan ke `figures/distribusi_label.png`.

In [3]:
TEXT_COL = "review"
LABEL_COL = "sentimen"
LABEL_NAMES = {0: "negatif", 1: "positif"}

if TEXT_COL not in df.columns:
    raise KeyError(f"Kolom teks '{TEXT_COL}' tidak ada. Kolom tersedia: {list(df.columns)}")
if LABEL_COL not in df.columns:
    raise KeyError(f"Kolom label '{LABEL_COL}' tidak ada. Kolom tersedia: {list(df.columns)}")

# Rekam metadata dataset SEBELUM kolom bantu ditambahkan
kolom_asli = [str(c) for c in df.columns]
tipe_awal = {str(c): str(t) for c, t in df.dtypes.items()}
missing_awal = {str(c): int(v) for c, v in df.isna().sum().items()}
duplikat_awal = int(df.duplicated().sum())
jumlah_data = int(len(df))

print("Jumlah data :", jumlah_data)
print("\nKolom dan tipe data:")
display(pd.DataFrame({"kolom": kolom_asli, "tipe_data": [tipe_awal[c] for c in kolom_asli]}))
print("\nMissing value per kolom:")
display(pd.DataFrame({"kolom": kolom_asli, "missing": [missing_awal[c] for c in kolom_asli]}))
print("Total baris duplikat penuh :", duplikat_awal)

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

label_counts = df[LABEL_COL].value_counts().sort_index()
label_pct = df[LABEL_COL].value_counts(normalize=True).sort_index() * 100
dist_df = pd.DataFrame({
    "label": label_counts.index,
    "jumlah": label_counts.values,
    "persentase": np.round(label_pct.values, 2),
})
dist_df["keterangan"] = dist_df["label"].map(lambda x: LABEL_NAMES.get(x, "lainnya"))
print("\nDistribusi label sentimen:")
display(dist_df)

fig, ax = plt.subplots(figsize=(6, 4))
warna = ["#d9534f" if int(l) == 0 else "#5cb85c" for l in dist_df["label"]]
bars = ax.bar(
    [f"{int(l)} ({LABEL_NAMES.get(l, 'lainnya')})" for l in dist_df["label"]],
    dist_df["jumlah"].values,
    color=warna,
)
for b, v, p in zip(bars, dist_df["jumlah"].values, dist_df["persentase"].values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{int(v)}\n({p:.2f}%)",
            ha="center", va="bottom", fontsize=9)
ax.set_title("Distribusi Label Sentimen Dataset Ulasan E-Commerce")
ax.set_ylabel("Jumlah ulasan")
ax.set_xlabel("Label sentimen")
ax.set_ylim(0, dist_df["jumlah"].max() * 1.18)
plt.tight_layout()
chart_path = FIG_DIR / "distribusi_label.png"
fig.savefig(chart_path, dpi=150)
plt.show()
print("Grafik disimpan ke:", chart_path)

Jumlah data : 11606

Kolom dan tipe data:


,kolom,tipe_data
0,review,str
1,translate,str
2,sentimen,int64



Missing value per kolom:


,kolom,missing
0,review,0
1,translate,0
2,sentimen,0


Total baris duplikat penuh : 4

Distribusi label sentimen:


,label,jumlah,persentase,keterangan
0,0,5803,50.0,negatif
1,1,5803,50.0,positif


Grafik disimpan ke: D:\NLP\preprocessing-ulasan-ecommerce-id\figures\distribusi_label.png


C:\Users\Hype AMD\AppData\Local\Temp\ipykernel_18360\3662433793.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Tahapan Preprocessing

Preprocessing dilakukan bertahap dan **setiap tahap disimpan pada kolom tersendiri**:

1. **Case folding** - mengubah seluruh teks menjadi huruf kecil agar "Bagus", "bagus", dan "BAGUS" dianggap sama.
2. **Cleaning** - menghapus URL, tag HTML, angka, tanda baca, karakter khusus/emoji, serta merapikan spasi berlebih.
3. **Tokenisasi** - memecah teks menjadi daftar kata (token) menggunakan NLTK, dengan fallback `split()` bila resource tokenizer tidak tersedia.
4. **Stopword removal** - membuang kata umum berbahasa Indonesia memakai daftar stopword **Sastrawi**.
5. **Stemming** - mengubah kata berimbuhan menjadi kata dasar memakai **Sastrawi**, dengan *cache* per kata unik agar tidak menghitung kata yang sama berulang kali.

Kolom hasil: `case_folding`, `cleaning`, `tokenisasi`, `stopword_removal`, dan `stemming` (hasil akhir).

In [4]:
import nltk

for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
stemmer = StemmerFactory().create_stemmer()

URL_RE = re.compile(r"http\S+|www\.\S+", re.IGNORECASE)
HTML_RE = re.compile(r"<[^>]+>")
NON_LETTER_RE = re.compile(r"[^a-z\s]")
WS_RE = re.compile(r"\s+")
WORD_RE = re.compile(r"[a-zA-Z]+")


def case_folding(text):
    return str(text).lower()


def cleaning(text):
    text = URL_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    text = NON_LETTER_RE.sub(" ", text)      # hapus angka, tanda baca, simbol, emoji
    text = WS_RE.sub(" ", text).strip()
    return text


def tokenize(text):
    if not text.strip():
        return []
    try:
        return nltk.word_tokenize(text)
    except Exception:
        return text.split()


def remove_stopwords(tokens):
    return [t for t in tokens if t and t not in STOPWORDS]


_stem_cache = {}


def stem_token(tok):
    if tok not in _stem_cache:
        _stem_cache[tok] = stemmer.stem(tok)
    return _stem_cache[tok]


def stem_tokens(tokens):
    return [stem_token(t) for t in tokens]


print("Jumlah stopword Sastrawi:", len(STOPWORDS))

t0 = time.time()
df["case_folding"] = df[TEXT_COL].map(case_folding)
df["cleaning"] = df["case_folding"].map(cleaning)
df["token_list"] = df["cleaning"].map(tokenize)
df["tokenisasi"] = df["token_list"].map(lambda ts: " ".join(ts))
df["stopword_list"] = df["token_list"].map(remove_stopwords)
df["stopword_removal"] = df["stopword_list"].map(lambda ts: " ".join(ts))
print("Melakukan stemming ...")
df["stem_list"] = df["stopword_list"].map(stem_tokens)
df["stemming"] = df["stem_list"].map(lambda ts: " ".join(ts))
print(f"Selesai dalam {time.time() - t0:.1f} detik")
print("Kata unik yang di-stem (ukuran cache):", len(_stem_cache))

display(df[[TEXT_COL, "case_folding", "cleaning", "tokenisasi", "stopword_removal", "stemming"]].head())

Jumlah stopword Sastrawi: 809


Melakukan stemming ...


Selesai dalam 4.5 detik
Kata unik yang di-stem (ukuran cache): 12045


,review,case_folding,cleaning,tokenisasi,stopword_removal,stemming
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara dan mic puas. hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara dan mic puas hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara dan mic puas hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara mic puas pesan yg nyala sy pastikan lg klik yg nyala tp yg ga nyala,suara mic puas pesan yg nyala sy pasti lg klik yg nyala tp yg ga nyala
1,Good. Mudah-mudahan awet. Makasih,good. mudah-mudahan awet. makasih,good mudah mudahan awet makasih,good mudah mudahan awet makasih,good mudah mudahan awet makasih,good mudah mudah awet makasih
2,"kondisi dus robek2, tolong di perhatikan lagi","kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek tolong di perhatikan lagi,kondisi dus robek tolong di perhatikan lagi,kondisi dus robek tolong perhatikan,kondisi dus robek tolong perhati
3,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo ...,kualitas produk baik. harga terjangkau dan pengirimannya cepet banget. udh 3x order disini oke lah tp sayangnya klo ...,kualitas produk baik harga terjangkau dan pengirimannya cepet banget udh x order disini oke lah tp sayangnya klo bua...,kualitas produk baik harga terjangkau dan pengirimannya cepet banget udh x order disini oke lah tp sayangnya klo bua...,kualitas produk harga terjangkau pengirimannya cepet banget udh order oke tp sayangnya klo kado gabisa nitip pake ka...,kualitas produk harga jangkau kirim cepet banget udh order oke tp sayang klo kado gabisa nitip pake kartu ucap bungk...
4,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit","overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit",overall bagus tombol berfungsi semuanya tapi packaging kurang karena sudah penyot dan robek sedikit,overall bagus tombol berfungsi semuanya tapi packaging kurang karena sudah penyot dan robek sedikit,overall bagus tombol berfungsi packaging penyot robek,overall bagus tombol fungsi packaging penyot robek


## 4. Contoh Sebelum dan Sesudah Preprocessing

Diambil **10 ulasan acak** dengan `random_state=42` agar hasil dapat direproduksi.
Tabel menampilkan teks asli dan hasil setiap tahap; indeks baris asli ikut disimpan di
`statistik.json` agar laporan PDF membaca contoh yang sama persis dari CSV.

In [5]:
contoh = df.sample(n=10, random_state=SEED).copy()
contoh_indeks = [int(i) for i in contoh.index]
contoh = contoh.reset_index(drop=False).rename(columns={"index": "index_asli"})
contoh.insert(0, "no", range(1, len(contoh) + 1))

show_cols = ["no", "index_asli", TEXT_COL, "case_folding", "cleaning",
             "tokenisasi", "stopword_removal", "stemming"]
display(contoh[show_cols])

print("\nContoh teks asli vs hasil akhir (dipotong 220 karakter untuk tampilan):")
for _, r in contoh.iterrows():
    print(f"[{int(r['no'])}] ASLI   : {r[TEXT_COL][:220]}")
    print(f"    BERSIH : {r['stemming'][:220]}")
    print("-" * 90)

,no,index_asli,review,case_folding,cleaning,tokenisasi,stopword_removal,stemming
0,1,8077,Barangnya original 👍. Bisa muat banyak. Gampang dipakai. Penjualnya kurang ramah,barangnya original 👍. bisa muat banyak. gampang dipakai. penjualnya kurang ramah,barangnya original bisa muat banyak gampang dipakai penjualnya kurang ramah,barangnya original bisa muat banyak gampang dipakai penjualnya kurang ramah,barangnya original muat gampang dipakai penjualnya ramah,barang original muat gampang pakai jual ramah
1,2,10959,Barang sudah sampai pengirimannya cepat,barang sudah sampai pengirimannya cepat,barang sudah sampai pengirimannya cepat,barang sudah sampai pengirimannya cepat,barang pengirimannya cepat,barang kirim cepat
2,3,5786,"realpict, packing oke","realpict, packing oke",realpict packing oke,realpict packing oke,realpict packing oke,realpict packing oke
3,4,2503,Cuman bisa komen 👍👍👍,cuman bisa komen 👍👍👍,cuman bisa komen,cuman bisa komen,cuman komen,cuman komen
4,5,10113,"mantap banget, pengiriman cepat pesan sekarang besok sudah ditangan.Desain elegan 😍. Resolusi gambar tajam 👍.","mantap banget, pengiriman cepat pesan sekarang besok sudah ditangan.desain elegan 😍. resolusi gambar tajam 👍.",mantap banget pengiriman cepat pesan sekarang besok sudah ditangan desain elegan resolusi gambar tajam,mantap banget pengiriman cepat pesan sekarang besok sudah ditangan desain elegan resolusi gambar tajam,mantap banget pengiriman cepat pesan besok ditangan desain elegan resolusi gambar tajam,mantap banget kirim cepat pesan besok tang desain elegan resolusi gambar tajam
5,6,4299,"awalnya dikrm hanya unit saja, setelah komplain baru dikrm dan info bonus bundling harusnya diupdate kalo tumbler da...","awalnya dikrm hanya unit saja, setelah komplain baru dikrm dan info bonus bundling harusnya diupdate kalo tumbler da...",awalnya dikrm hanya unit saja setelah komplain baru dikrm dan info bonus bundling harusnya diupdate kalo tumbler dah...,awalnya dikrm hanya unit saja setelah komplain baru dikrm dan info bonus bundling harusnya diupdate kalo tumbler dah...,dikrm unit komplain dikrm info bonus bundling diupdate kalo tumbler dah habis pelanggan kecewa,dikrm unit komplain dikrm info bonus bundling diupdate kalo tumbler dah habis langgan kecewa
6,7,10333,"menurut infonya free Mola TV tp ternyata perlu bayar, setelah diskusi panjang akhirnya diketahui kalau OS nya tidak ...","menurut infonya free mola tv tp ternyata perlu bayar, setelah diskusi panjang akhirnya diketahui kalau os nya tidak ...",menurut infonya free mola tv tp ternyata perlu bayar setelah diskusi panjang akhirnya diketahui kalau os nya tidak d...,menurut infonya free mola tv tp ternyata perlu bayar setelah diskusi panjang akhirnya diketahui kalau os nya tidak d...,infonya free mola tv tp bayar diskusi os diupdate pembelian tv terima,info free mola tv tp bayar diskusi os diupdate beli tv terima
7,8,4877,Pesanan sesuai cuma syang nya gak ada saringanya buat mengkukus lain kali lebih teliti lgi ya,pesanan sesuai cuma syang nya gak ada saringanya buat mengkukus lain kali lebih teliti lgi ya,pesanan sesuai cuma syang nya gak ada saringanya buat mengkukus lain kali lebih teliti lgi ya,pesanan sesuai cuma syang nya gak ada saringanya buat mengkukus lain kali lebih teliti lgi ya,pesanan sesuai syang gak saringanya mengkukus teliti lgi,pesan sesuai syang gak saringanya kukus teliti lgi
8,9,8239,remote control tidak peka,remote control tidak peka,remote control tidak peka,remote control tidak peka,remote control peka,remote control peka
9,10,7078,Sudah sampai mesin cuci nya terima kasih cepat ya 5 hari sampai...kerennn,sudah sampai mesin cuci nya terima kasih cepat ya 5 hari sampai...kerennn,sudah sampai mesin cuci nya terima kasih cepat ya hari sampai kerennn,sudah sampai mesin cuci nya terima kasih cepat ya hari sampai kerennn,mesin cuci terima kasih cepat kerennn,mesin cuci terima kasih cepat kerennn



Contoh teks asli vs hasil akhir (dipotong 220 karakter untuk tampilan):
[1] ASLI   : Barangnya original 👍. Bisa muat banyak. Gampang dipakai. Penjualnya kurang ramah
    BERSIH : barang original muat gampang pakai jual ramah
------------------------------------------------------------------------------------------
[2] ASLI   : Barang sudah sampai pengirimannya cepat
    BERSIH : barang kirim cepat
------------------------------------------------------------------------------------------
[3] ASLI   : realpict, packing oke
    BERSIH : realpict packing oke
------------------------------------------------------------------------------------------
[4] ASLI   : Cuman bisa komen 👍👍👍
    BERSIH : cuman komen
------------------------------------------------------------------------------------------
[5] ASLI   : mantap banget, pengiriman cepat pesan sekarang besok sudah ditangan.Desain elegan 😍. Resolusi gambar tajam 👍.
    BERSIH : mantap banget kirim cepat pesan besok tang desain elegan reso

## 5. Statistik Dampak Preprocessing

Bagian ini mengukur dampak preprocessing secara kuantitatif:

- rata-rata panjang teks (jumlah karakter dan jumlah kata) sebelum vs sesudah,
- ukuran *vocabulary* (jumlah kata unik) sebelum vs sesudah beserta persentase penurunannya,
- jumlah teks yang menjadi kosong setelah preprocessing.

Nilai-nilai ini disimpan ke `data/statistik.json`, sedangkan seluruh data hasil preprocessing
disimpan ke `data/hasil_preprocessing.csv`.

In [6]:
def word_count_raw(s):
    return len(WORD_RE.findall(str(s)))


def vocab_raw(series):
    v = set()
    for s in series:
        v.update(w.lower() for w in WORD_RE.findall(str(s)))
    return v


raw_texts = df[TEXT_COL].astype(str)
final_texts = df["stemming"].astype(str)

avg_char_before = float(raw_texts.str.len().mean())
avg_char_after = float(final_texts.str.len().mean())
avg_word_before = float(raw_texts.map(word_count_raw).mean())
avg_word_after = float(final_texts.map(lambda s: len(s.split())).mean())
total_word_before = int(raw_texts.map(word_count_raw).sum())
total_word_after = int(final_texts.map(lambda s: len(s.split())).sum())

vocab_before = vocab_raw(raw_texts)
vocab_after = set(w for s in final_texts for w in s.split() if w)
penurunan_vocab = len(vocab_before) - len(vocab_after)
penurunan_vocab_persen = (penurunan_vocab / len(vocab_before) * 100) if vocab_before else 0.0
jumlah_kosong = int(final_texts.str.strip().eq("").sum())

ringkasan = pd.DataFrame([
    ["Rata-rata panjang karakter", round(avg_char_before, 2), round(avg_char_after, 2)],
    ["Rata-rata panjang kata", round(avg_word_before, 2), round(avg_word_after, 2)],
    ["Total kata", total_word_before, total_word_after],
    ["Ukuran vocabulary (kata unik)", len(vocab_before), len(vocab_after)],
], columns=["Metrik", "Sebelum", "Sesudah"])
display(ringkasan)
print(f"Penurunan vocabulary: {penurunan_vocab} kata ({penurunan_vocab_persen:.2f}%)")
print(f"Teks kosong setelah preprocessing: {jumlah_kosong} dari {len(df)} "
      f"({jumlah_kosong / len(df) * 100:.2f}%)")

# Simpan CSV hasil preprocessing
out_cols = [TEXT_COL, LABEL_COL, "case_folding", "cleaning", "tokenisasi",
            "stopword_removal", "stemming"]
csv_path = DATA_DIR / "hasil_preprocessing.csv"
df[out_cols].to_csv(csv_path, index=False, encoding="utf-8")
print("CSV disimpan ke:", csv_path)

statistik = {
    "nama": "Muhammad Hafidz",
    "nim": "24146028",
    "github": "https://github.com/MuhammadHafidz212-cell/nlp",
    "tanggal": datetime.now().strftime("%d-%m-%Y"),
    "dataset": {
        "nama": DATASET_NAME,
        "sumber": "https://huggingface.co/datasets/dipawidia/ecommerce-product-reviews-sentiment",
        "metode_load": load_method,
        "jumlah_data": jumlah_data,
        "kolom": kolom_asli,
        "tipe_data": tipe_awal,
        "missing": missing_awal,
        "duplikat": duplikat_awal,
        "label_kolom": LABEL_COL,
        "distribusi_label": {
            str(int(l)): {
                "keterangan": LABEL_NAMES.get(int(l), "lainnya"),
                "jumlah": int(n),
                "persentase": round(float(p), 2),
            }
            for l, n, p in zip(dist_df["label"], dist_df["jumlah"], dist_df["persentase"])
        },
    },
    "contoh_indeks": contoh_indeks,
    "contoh": [
        {
            "no": int(r["no"]),
            "index_asli": int(r["index_asli"]),
            "review": r[TEXT_COL],
            "case_folding": r["case_folding"],
            "cleaning": r["cleaning"],
            "tokenisasi": r["tokenisasi"],
            "stopword_removal": r["stopword_removal"],
            "stemming": r["stemming"],
        }
        for _, r in contoh.iterrows()
    ],
    "statistik": {
        "rata_panjang_karakter_sebelum": round(avg_char_before, 2),
        "rata_panjang_karakter_sesudah": round(avg_char_after, 2),
        "rata_panjang_kata_sebelum": round(avg_word_before, 2),
        "rata_panjang_kata_sesudah": round(avg_word_after, 2),
        "total_kata_sebelum": total_word_before,
        "total_kata_sesudah": total_word_after,
        "vocabulary_sebelum": len(vocab_before),
        "vocabulary_sesudah": len(vocab_after),
        "penurunan_vocabulary": penurunan_vocab,
        "penurunan_vocabulary_persen": round(penurunan_vocab_persen, 2),
        "jumlah_teks_kosong": jumlah_kosong,
        "persentase_teks_kosong": round(jumlah_kosong / len(df) * 100, 2),
        "jumlah_stopword_sastrawi": len(STOPWORDS),
    },
}

json_path = DATA_DIR / "statistik.json"
json_path.write_text(json.dumps(statistik, ensure_ascii=False, indent=2), encoding="utf-8")
print("JSON disimpan ke:", json_path)

,Metrik,Sebelum,Sesudah
0,Rata-rata panjang karakter,103.57,65.76
1,Rata-rata panjang kata,15.74,10.82
2,Total kata,182713.00,125602.00
3,Ukuran vocabulary (kata unik),12558.00,10036.00


Penurunan vocabulary: 2522 kata (20.08%)
Teks kosong setelah preprocessing: 20 dari 11606 (0.17%)
CSV disimpan ke: D:\NLP\preprocessing-ulasan-ecommerce-id\data\hasil_preprocessing.csv


JSON disimpan ke: D:\NLP\preprocessing-ulasan-ecommerce-id\data\statistik.json


## 6. Kesimpulan

Preprocessing **menurunkan noise** (URL, angka, tanda baca, emoji, kata umum) sehingga teks
lebih bersih dan seragam. Case folding dan cleaning **menormalkan variasi penulisan**, sedangkan
stemming **menyatukan bentuk berimbuhan** ke kata dasar sehingga ukuran vocabulary mengecil dan
fitur teks menjadi lebih padat. Dampak positif ini harus dibaca bersama risikonya: stopword
removal Sastrawi dapat membuang kata negasi seperti **"tidak"**, dan stemming dapat keliru pada
kata tertentu (mis. singkatan atau kata tidak baku), sehingga makna ulasan bisa bergeser.
Untuk analisis sentimen, penghapusan negasi sebaiknya dikaji ulang (mis. mempertahankan kata
negasi) agar polaritas tidak berubah.

In [7]:
s = statistik["statistik"]
print("RINGKASAN DAMPAK PREPROCESSING")
print(f"- Rata-rata panjang karakter : {s['rata_panjang_karakter_sebelum']} -> {s['rata_panjang_karakter_sesudah']}")
print(f"- Rata-rata panjang kata     : {s['rata_panjang_kata_sebelum']} -> {s['rata_panjang_kata_sesudah']}")
print(f"- Vocabulary                 : {s['vocabulary_sebelum']} -> {s['vocabulary_sesudah']} "
      f"(turun {s['penurunan_vocabulary_persen']}%)")
print(f"- Teks kosong setelah proses : {s['jumlah_teks_kosong']} ({s['persentase_teks_kosong']}%)")
print("\nFile keluaran:")
print(" -", DATA_DIR / "hasil_preprocessing.csv")
print(" -", DATA_DIR / "statistik.json")
print(" -", FIG_DIR / "distribusi_label.png")

RINGKASAN DAMPAK PREPROCESSING
- Rata-rata panjang karakter : 103.57 -> 65.76
- Rata-rata panjang kata     : 15.74 -> 10.82
- Vocabulary                 : 12558 -> 10036 (turun 20.08%)
- Teks kosong setelah proses : 20 (0.17%)

File keluaran:
 - D:\NLP\preprocessing-ulasan-ecommerce-id\data\hasil_preprocessing.csv
 - D:\NLP\preprocessing-ulasan-ecommerce-id\data\statistik.json
 - D:\NLP\preprocessing-ulasan-ecommerce-id\figures\distribusi_label.png
